# DEMONEXT CCD Camera Testing Sandbox

This notebook is a sandbox for running tests of the DEMONEXT CCD science camera
with the demonext `Camera` class.

## WARNING: READ THIS

**Never, Ever, Run All in this notebook at once**  

This was designed to test individula telescope operations and document them. Run without care you could
try to drive the telescope places you don't want it to go.

In [ ]:
import sys
import os
import time
import math
import glob
import datetime
from datetime import date, timedelta

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, coordinates, and times

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import TETE
from astropy.time import Time

# we use pathlib for platform-agnostic path handling 

from pathlib import Path

# we use YAML for runtime configuration files

import yaml

# We use logging for runtime logs

import logging

# demonext module

import demonext
from demonext import config, telescope, pdu, camera, focuser, obsfile, site

# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}


## Load the runtime configuration file

DEMONEXT uses a YAML-formatted runtime configuration file kept in a common directory.

In [ ]:
# platform-agnostic path definition relative to home

configDir = Path.home() / ".demonext/config"
configFile = "demonext.txt"

cfgFile = str(Path.home() / configDir / configFile)

# read by instantiating a demonext Config class

try:
    cfg = config.Config(cfgFile)
    print("Runtime configuration file loaded")
except Exception as exp:
    print(f"ERROR: (Config) - {exp}")

# start the runtime logger

logDir = demonext.homePath(cfg.config["directories"]["LogDir"])

logFile = str(Path(logDir) / f"eng{demonext.obsDate()}.txt")

logging.basicConfig(filename=logFile,
                    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
                    filemode="a",
                    level=logging.INFO)

# ID for log entries

logger = logging.getLogger("camNotes")

logger.info("Started the DEMONEXT camera control development notebook")

## DEMONEXT `Camera` class

instantiate camera and telescope classes

In [ ]:
# instantiate Telescope, Camera, and Focuser class instances

# Camera class 

try:
    cam = camera.Camera(cfgFile)
    print("Camera class started")
except Exception as exp:
    msg = f"Cannot initialize Camera instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Telescope class

try:
    tel = telescope.Telescope(cfgFile)
    print("Telescope class started")
except Exception as exp:
    msg = f"Cannot initialize Telescope instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Focuser class

try:
    foc = focuser.Focuser(cfgFile)
    print("Focuser class started")
except Exception as exp:
    msg = f"Cannot initialize Focuser instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Site class

try:
    sro = site.Site(cfgFile)
    print("Site class started")
except Exception as exp:
    msg = f"Cannot initialize a Site instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# share the Telescope class object with the Camera class instance

cam.telescope = tel

# share the Focuser class object with the Camera class instance

cam.focuser = foc

# Share the Site class object with the Camera class object 

cam.site = sro

# Connect the subsystems (telescope mount, cameras, focuser)

# connect to the telescope controller

logger.info("Connecting to telescope...")

print("Connecting to the telescope mount...")
try:
    tel.connect()
except Exception as exp:
    print(f"Cannot connect to telescope: {exp}")

if tel.connected:
    print(f"Done: Connected to {tel.telName} successfully")
else:
    print(f"Error: Failed to connect to the telescope controller")
    
# connect the camera

print("\nConnecting to the cameras...")
try:
    cam.connect()
except Exception as exp:
    print(f"Cannot connect to the cameras: {exp}")

if cam.connected:
    print(f"Done: Connected to MaxIM DL and the cameras")
else:
    print(f"Error: Failed to connect to MaxIM DL and the cameras")

# connect the focuser

print("\nConnecting to the focuser...")
try:
    foc.connect()
except Exception as exp:
    print(f"Cannot connect to the focuser: {exp}")

if foc.connected:
    print(f"Done: Connected to {foc.name}")
else:
    print(f"Error: Failed to connect to PWI3 and the Hedrick focusers")


## Get camera and telescope information


In [ ]:
# query mount information only, no moves yet

logger.info("querying CCD info")

ccdInfo = cam.getCCDInfo()
try:
    ccdInfo = cam.getCCDInfo()
except Exception as exp:
    print(f"Cannot get camera info: {exp}")

print("Science CCD Info:")
print(f"  Format: {ccdInfo['sizeX']}x{ccdInfo['sizeY']}")
print(f"  Pixel Size: {ccdInfo['pixSizeX']}x{ccdInfo['pixSizeY']}")
print(f"  Binning: {ccdInfo['binX']}x{ccdInfo['binY']}")
ccdXS = ccdInfo['startX']
ccdXE = ccdXS + ccdInfo['numX']
ccdYS = ccdInfo['startY']
ccdYE = ccdYS + ccdInfo['numY']
print(f"  Readout ROI: [{ccdXS}:{ccdXE},{ccdYS}:{ccdYE}]")

print("\nCCD Thermoelectric Cooler Status:")
print(f"     Cooler State: {ccdInfo['TECState']}, Cooler Power: {ccdInfo['TECPower']:.1f}%")
print(f"  CCD Temperature: {ccdInfo['CCDTemp']:.2f}C")
print(f"   Heat Sink Temp: {ccdInfo['TECTemp']:.2f}C")
print(f"    SetPoint Temp: {ccdInfo['setPoint']:.2f}C")
    
# telescope info

try:
    telInfo = tel.position()
except Exception as exp:
    print(f"Cannot get telescope position info: {exp}")

print(f"\nTelescope Position (decimal):")
print(f"  Alt/Az: {telInfo["Alt"]:.5f} d, {telInfo["Az"]:.5f} d")
print(f"  RA/Dec: {telInfo["RA"]:.5f} h, {telInfo["Dec"]:.5f} d")
print(f"  LST/HA: {telInfo["LST"]:.5f}h, {telInfo["HA"]:.5f} h")
print(f"    SecZ: {telInfo["SecZ"]:.2f}")

telFITS = tel.telFITS()
print(f"\nTelescope Position (FITS style):")
print(f"  Alt/Az: {telFITS["TELALT"]} {telFITS["TELAZ"]}")
print(f"  RA/Dec: {telFITS["TELRA"]} {telFITS["TELDEC"]}")
print(f"  LST/HA: {telFITS["LST"]} {telFITS["TELHA"]}")
print(f"    SecZ: {telFITS["SECZ"]:.2f}")

# quick tests of the decimal <-> sexigesimal conversion methods 

print("\nFormat Conversion:")
print(f"  sexigesimal -> decimal: {telFITS["TELRA"]} = {tel.sex2dec(telFITS["TELRA"]):.8f}")
print(f"  decimal -> sexigesimal: {telInfo["RA"]:.8f} = {tel.dec2sex(telInfo["RA"])}")

# Mount Status

print(f"\nTelescope Mount Status:")
print(f"   At Home? {YesNo[tel.isHome()]}")
print(f"    Parked? {YesNo[tel.isParked()]}")
print(f"  Tracking: {OnOff[tel.isTracking()]}")
print(f"   Slewing? {YesNo[tel.isSlewing()]}")

# Pointing limits

print(f"\nPointing Limits:")
print(f"   HA: {tel.minHA:.1f} to {tel.maxHA:.1f}h")
print(f"  Dec: {tel.minDec:.2f} to 0 d")
print(f"  Alt: {tel.minAlt:.1f} to 90 d")

# data directory info

print("\nData Directories and Filenames:")
print(f"  Base Data Directory: {cam.setDataDir()}")
print(f"  Observing Date: {cam.getObsDate()}")
for imgType in ['sci','bias','dark','flat','gcal']:
    print(f"  Next {imgType} file: {cam.getNextFile(imgType)}")

# direct cooling methods

print("\nCooling Info direct methods:")
print(f"  CCD Temp = {cam.ccdTemp():.2f} C")
print(f"  TEC Temp = {cam.tecTemp():.2f} C")
print(f"  SetPoint = {cam.setPoint():.1f} C")
print(f" TEC Power = {cam.tecPower()}%")
print(f"   Cooling: {cam.isCooling()} = {OnOff[cam.isCooling()]}")

# Focuser status

print(f"{foc.name} Info:")
focInfo = foc.getFocInfo()
for key,val in focInfo.items():
    print(f"  {key}: {val}")

print(f"\nCurrent focus position: {foc.getPos()} microns")
pmTemp,ambTemp = foc.getTemps()
print(f"  Primary mirror temperature: {pmTemp:.1f} C")
print(f"  Ambient air temperature: {ambTemp:.1f} C")

print(f"  Maximum Focus: {foc.maxFocus:.0f} microns")
print(f"  Focus Timeout: {foc.focusTimeout:.1f} seconds")

# Site info

# Sun altitude relative to the local horizon

sunPos = sro.sunAlt()
print(f"Sun altitude: {sunPos:.4f} deg")

# Day or Night?

if sro.isNight():
    print("Night: the Sun is below the horizon at SRO")
    if sro.isDark():
        print("       it is dark (sun >18-deg below the horizon)")
    else:
        if sro.isTwilight():
            print("       it is twilight (12 < sun < 18-deg below the horizon)")
        else:
            print("       it is dusk (sun < 12-deg below the horizon)")
else:
    print("  Day: the Sun is above the horizon at SRO")

# Roof status

if sro.roofOpen():
    print("\nThe SRO Building 14 roof is OPEN")
else:
    if sro.roof["position"] == "UNKNOWN":
        print(f"\nSRO Building 14 roof position is UNKNOWN - {sro.msg}")
    else:
        print("\nThe SRO Building 14 roof is CLOSED")

# Weather station query

if sro.getWeather():
    if sro.weather['up']:
        print(f"\nSRO Weather at {sro.weather['iso']} PT:")
        print(f"      Temp: {sro.weather['airTemp']:.1f} C")
        print(f"  Humidity: {sro.weather['humidity']:.1f}%  dewPoint: {sro.weather['dewpoint']:.1f} C")
        print(f"      Wind: {sro.weather['winds']}, {sro.weather['windspeed']:.1f} m/s")
        print(f"       Sky: {sro.weather['clouds']}, {sro.weather['rain']}, {sro.weather['sky']}, Temp: {sro.weather['skyTemp']:.1f} C")
else:
    print(f"\nSRO weather station not responding - {sro.msg}")

## Camera cooldown

Enable camera cooling and wait until it reaches the setpoint temperature.  Time how long
it takes to reach the operating setpoint.

In [ ]:
# cool the camera to the setpoint

print("Start:")
print(f" CCD Temp: {cam.ccdTemp():.2f} C")
print(f" TEC Temp: {cam.tecTemp():.2f} C")
print(f"TEC Power: {cam.tecPower():.1f}% ")
print(f" SetPoint: {cam.setPoint():.1f} C\n")

print(f"Cooling down the CCD camera...")

t0 = time.time()
try:
    cam.cooldown()
except Exception as exp:
    print(f"oops: {exp}")

print(f"Done: CCD camera temperature is {cam.ccdTemp():.1f}C")
dt = (time.time() - t0)/60.0
print(f"   cooldown time: {dt:.1f} minutes")
print(f"        CCD Temp: {cam.ccdTemp():.2f} C")
print(f"        TEC Temp: {cam.tecTemp():.2f} C")
print(f"       TEC Power: {cam.tecPower():.1f}% ")


## Acquire an image

Here we go...

Take images with the camera and make sure they go where they should. End-to-end test of the basic image acquisition process.

In [ ]:
expFilt = "r_SDSS"
expTime = 1.0
print(f"Taking a science image, expTime={expTime:.1f} sec, with the {expFilt} filter...")
t0 = time.time()
try:
    fnum = cam.filterNum(expFilt)
    cam.acquire("sci",expTime,fnum)
except Exception as exp:
    print(f"oops: {exp}")
print(f"Done: wrote {cam.nextFile} to disk")
dt = time.time() - t0
print(f"      execution time: {dt:.3f} seconds")

### Take a series of bias images

The `bias()` method takes an optional argument, `nimgs=` which sets the number of bias
images to acquire.  Because bias images do not require guiding, tracking, or a filter,
this allows us to take multiple images with minimal setup and acquisition overheads.

Default is 1 image.

In [ ]:
numBias = 5
print(f"Taking {numBias} bias images...")
t0 = time.time()
try:
    cam.bias(nimgs=numBias)
except Exception as exp:
    print(f"oops: {exp}")
print(f"Done: last bias image {cam.nextFile}")
dt = time.time() - t0
print(f"      execution time: {dt:.3f} seconds")

### Take dark images

The `dark()` method takes an optional argument, `nimgs=` which sets the number of dark
images to acquire.  Because darks do not require guiding, tracking, or a filter,
this allows us to take multiple images with minimal setup and acquisition overheads.

Default is 1 image.

In [ ]:
numDarks = 5
expTime = 300.0
print(f"Taking {numDarks} {expTime:.1f} sec dark images...")
t0 = time.time()
try:
    cam.dark(expTime,nimgs=numDarks)
except Exception as exp:
    print(f"oops: {exp}")
print(f"Done: last dark image {cam.nextFile}")
dt = time.time() - t0
print(f"      execution time: {dt:.3f} seconds")

### Dark Test

Use this cell for dark sequences, varies by test.

In [ ]:
numDarks = 5
numBias = 5
expTimes = [30,60,90,120,150,180,240,300]
for expTime in expTimes:
    print(f"\nDark sequence for {int(expTime)} sec...")
    try:
        cam.bias(nimgs=numBias)
    except Exception as exp:
        print(f"oops: {exp}")
    try:
        cam.dark(expTime,nimgs=numDarks)
    except Exception as exp:
        print(f"oops: {exp}")

# ending bias

try:
    cam.bias(nimgs=numBias)
except Exception as exp:
    print(f"oops: {exp}")

print("\nDark test run complete")

### Take a series of science images

The `science()` method is for images guided using the guide telescope.  Science guiding is done with a
different procedure.

`science()` can take 1, 2, or 3 arguments:
 * `science(obsDef)` - exposure parameters are in an `obsDef` observation definition list (see below)
 * `science(filtNum,expTime)` - take one image through filter number `filtNum` of `expTime` seconds
 * `science(filtNum,expTime,nimgs)` - take `nimgs` images with the given filter and exposure time.

The `obsDef` list is in the format returned from a YAML-format observation description file.  It ha
the format
 > obsDef = [filtID,expTime,nimgs]

where:
 * `filtID` : string with the filter ID
 * `expTime` : float with the exposure time in seconds
 * `nimgs` : integer with the number of exposures to acquire

example:
 > obsDef = ['r_SDSS',10.0,2]

will take 2 10s images through the r_SDSS filter.

In [ ]:
obsDef = ['g_SDSS',5.0,3]
print(f"Taking {obsDef[2]} science images of {obsDef[1]:.1f} sec through the {obsDef[0]} filter...")
t0 = time.time()
try:
    cam.science(obsDef)
except Exception as exp:
    print(f"oops: {exp}")

print(f"Done: last science image {cam.nextFile}")
dt = time.time() - t0
print(f"      execution time: {dt:.3f} seconds")

In [ ]:
filtID = 'r_SDSS'
filtNum = cam.filterNum(filtID)
expTime = 5.0
nimgs = 2

print(f"Taking {nimgs} science images of {expTime:.1f} sec through the {filtID} filter...")
t0 = time.time()
try:
    cam.science(expTime,filtNum,nimgs)
except Exception as exp:
    print(f"oops: {exp}")

print(f"Done: last science image {cam.nextFile}")
dt = time.time() - t0
print(f"      execution time: {dt:.3f} seconds")

## Camera warm-up

Controlled warm-up of the the camera to ambient.

In [ ]:
# warm the camera to ambient

t0 = time.time()
print(f"Warming the CCD camera to ambient...")

try:
    cam.warmup()
except Exception as exp:
    print(f"oops: {exp}")

print(f"Done: CCD camera temperature is {cam.ccdTemp():.1f}C")
dt = (time.time() - t0)/60.0
print(f"      warmup time: {dt:.1f} minutes")
